In [2]:
# pip 최신버전 업그레이드
!python -m pip install --upgrade pip

In [ ]:
# neo4j 관련 라이브러리 설치
%pip install neo4j langchain-neo4j


   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   -------- ------------------------------- 1/5 [neo4j]
   ---------------- ----------------------- 2/5 [json-repair]
   ---------------- ----------------------- 2/5 [json-repair]
   ---------------- ---------------

In [4]:
import os
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [7]:
load_dotenv() # .env 파일을 환경변수로 등록

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')


In [ ]:
# Neo4j Driver 생성
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

driver.verify_connectivity() # 실제로 연결됬는지 확인

print("Neo4j Driver 연결 성공!")


Neo4j Driver 연결 성공!


In [ ]:
# Python에서 Cypher 실행 (Student 노드 확인)
query = """ 
MATCH (student:Student)
RETURN
    student.student_id AS student_id,
    student.name AS name,
    student.age AS age
ORDER BY student.student_id    
"""

# 쿼리 결과 반환 (실제 조회 결과 행들, 쿼리 실행 요약 정보, 결과 컬럼명)
records, summary, keys = driver.execute_query(query, database = NEO4J_DATABASE)

student = [records.data() for records in records]

student

[{'student_id': 1, 'name': '홍길동', 'age': 26},
 {'student_id': 2, 'name': '김영희', 'age': 28},
 {'student_id': 3, 'name': '이민수', 'age': 24},
 {'student_id': 4, 'name': '박서연', 'age': 27},
 {'student_id': 5, 'name': '최준호', 'age': 30}]

In [11]:
student_df = pd.DataFrame(student)
student_df

,student_id,name,age
0,1,홍길동,26
1,2,김영희,28
2,3,이민수,24
3,4,박서연,27
4,5,최준호,30


In [12]:
# 파라미터를 이용한 조회
query = """
MATCH (student:Student {name: $student_name}) - [enroll:ENROLLED_IN] -> (course:Course)
RETURN
    student.name AS student_name,
    course.name AS course_name,
    enroll.score AS score,
    course.course_id AS course_id    
ORDER BY course_id
"""

records, summary, keys = driver.execute_query(
    query,
    student_name = '홍길동', 
    database = NEO4J_DATABASE
)

result = [records.data() for records in records]

result_df = pd.DataFrame(result)

result_df

,student_name,course_name,score,course_id
0,홍길동,Python,95,101
1,홍길동,Data Analysis,90,104


In [13]:
# Python에서 Cypher 실행 (Student 노드 확인)
query = """ 
MATCH
    (student:Student)
    -[:ENROLLED_IN]->
    (course:Course)
    <-[:TEACHES]-
    (instructor:Instructor)

MATCH
    (course)-[:BELONGS_TO]->(category:Category)

RETURN
    student.name AS student_name,
    course.name AS course_name,
    instructor.name AS instructor_name,
    category.name AS category_name,
    student.student_id AS student_id,
    course.course_id AS course_id

ORDER BY student_id, course_id  
"""

# 쿼리 결과 반환 (실제 조회 결과 행들, 쿼리 실행 요약 정보, 결과 컬럼명)
records, summary, keys = driver.execute_query(query, database = NEO4J_DATABASE)

result = [records.data() for records in records]

result_df = pd.DataFrame(result)
result_df

,student_name,course_name,instructor_name,category_name,student_id,course_id
0,홍길동,Python,Capybara,프로그래밍,1,101
1,홍길동,Data Analysis,Alice,데이터분석,1,104
2,김영희,Database,Capybara,데이터베이스,2,102
3,김영희,Machine Learning,Alice,인공지능,2,103
4,이민수,Python,Capybara,프로그래밍,3,101
5,이민수,Data Analysis,Alice,데이터분석,3,104
6,박서연,Machine Learning,Alice,인공지능,4,103
7,박서연,Deep Learning,Bob,인공지능,4,105
8,최준호,Database,Capybara,데이터베이스,5,102
9,최준호,Langchain,Bob,인공지능,5,106


In [14]:
# 여러단계의 관계 조회
query ='''
MATCH (student:Student)-[:ENROLLED_IN]->(course:Course)
RETURN
    course.name AS course_name,
    count(student) AS student_count
ORDER BY student_count DESC, course_name
'''

# 쿼리 결과 반환 (실제 조회 결과 행들, 쿼리 실행 요약 정보, 결과 컬럼명)
records, summary, keys = driver.execute_query(query, database= NEO4J_DATABASE)

result = [record.data() for record in records]

result_df = pd.DataFrame(result)

result_df

,course_name,student_count
0,Data Analysis,2
1,Database,2
2,Machine Learning,2
3,Python,2
4,Deep Learning,1
5,Langchain,1


In [15]:
driver.close()